Index - KIC-BSCCSAI-251P-053
Name – M H M MUNSHIF

Phase 2: Technical Application — Baseline Implementation 

Introduction and objective


In [8]:
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    confusion_matrix,
    balanced_accuracy_score,
    f1_score,
)

RANDOM_STATE = 42
TEST_SIZE = 0.20

Data loading and exploratory checks


In [9]:
DATA_PATH = Path("csv_result-Phishing_Legitimate_full.csv")
TARGET_COLUMN = "CLASS_LABEL"

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_PATH.resolve()}. "
        "Place csv_result-Phishing_Legitimate_full.csv beside this notebook or update DATA_PATH."
    )

df = pd.read_csv(DATA_PATH)
print(f"Original dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

if TARGET_COLUMN not in df.columns:
    raise ValueError(
        f"Target column '{TARGET_COLUMN}' was not found. "
        f"Available columns: {list(df.columns)}"
    )

print(f"Exact duplicate rows (including target): {df.duplicated().sum()}")
print("\nTarget distribution (original data):")
print(df[TARGET_COLUMN].value_counts(dropna=False))

Original dataset shape: (10000, 50)
Columns: ['id', 'NumDots', 'SubdomainLevel', 'PathLevel', 'UrlLength', 'NumDash', 'NumDashInHostname', 'AtSymbol', 'TildeSymbol', 'NumUnderscore', 'NumPercent', 'NumQueryComponents', 'NumAmpersand', 'NumHash', 'NumNumericChars', 'NoHttps', 'RandomString', 'IpAddress', 'DomainInSubdomains', 'DomainInPaths', 'HttpsInHostname', 'HostnameLength', 'PathLength', 'QueryLength', 'DoubleSlashInPath', 'NumSensitiveWords', 'EmbeddedBrandName', 'PctExtHyperlinks', 'PctExtResourceUrls', 'ExtFavicon', 'InsecureForms', 'RelativeFormAction', 'ExtFormAction', 'AbnormalFormAction', 'PctNullSelfRedirectHyperlinks', 'FrequentDomainNameMismatch', 'FakeLinkInStatusBar', 'RightClickDisabled', 'PopUpWindow', 'SubmitInfoToEmail', 'IframeOrFrame', 'MissingTitle', 'ImagesOnlyInForm', 'SubdomainLevelRT', 'UrlLengthRT', 'PctExtResourceUrlsRT', 'AbnormalExtFormActionR', 'ExtMetaScriptLinkRT', 'PctExtNullSelfRedirectHyperlinksRT', 'CLASS_LABEL']
Exact duplicate rows (including tar

Deduplication and contradictory-label handling


In [10]:
df_dedup = df.drop_duplicates().copy()
print(f"After exact-row deduplication: {df_dedup.shape}")

X_all = df_dedup.drop(columns=[TARGET_COLUMN])
y_all = df_dedup[TARGET_COLUMN]

label_counts_per_features = (
    df_dedup.groupby(list(X_all.columns), dropna=False)[TARGET_COLUMN]
    .nunique(dropna=False)
)
contradictory_feature_groups = label_counts_per_features[
    label_counts_per_features > 1
].index

if len(contradictory_feature_groups) > 0:
    feature_index = pd.MultiIndex.from_frame(X_all)
    contradictory_mask = feature_index.isin(contradictory_feature_groups)
    X_clean = X_all.loc[~contradictory_mask].copy()
    y_clean = y_all.loc[~contradictory_mask].copy()
else:
    X_clean = X_all.copy()
    y_clean = y_all.copy()

print(f"Unique feature vectors with contradictory labels: {len(contradictory_feature_groups)}")
print(f"Rows removed due to contradictory feature vectors: {len(X_all) - len(X_clean)}")
print(f"Final modeling data shape: {X_clean.shape}")
print("\nTarget distribution after cleaning:")
print(y_clean.value_counts(dropna=False))

if X_clean.empty:
    raise ValueError("No rows remain after cleaning; review the dataset and target column.")

non_numeric = X_clean.select_dtypes(exclude=[np.number]).columns.tolist()
if non_numeric:
    raise ValueError(
        f"Non-numeric feature columns found: {non_numeric}. "
        "Encode or remove these columns before modeling."
    )

if X_clean.isna().any().any() or y_clean.isna().any():
    raise ValueError(
        "Missing values detected. Add an imputation step to the pipeline or clean the dataset."
    )

After exact-row deduplication: (10000, 50)
Unique feature vectors with contradictory labels: 0
Rows removed due to contradictory feature vectors: 0
Final modeling data shape: (10000, 49)

Target distribution after cleaning:
CLASS_LABEL
1    5000
0    5000
Name: count, dtype: int64


Class balance, splitting, and preprocessing


In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X_clean,
    y_clean,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y_clean,
)

print(f"Training rows: {len(X_train)}")
print(f"Test rows: {len(X_test)}")
print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True).round(3))
print("\nTest target distribution:")
print(y_test.value_counts(normalize=True).round(3))

Training rows: 8000
Test rows: 2000

Training target distribution:
CLASS_LABEL
0    0.5
1    0.5
Name: proportion, dtype: float64

Test target distribution:
CLASS_LABEL
0    0.5
1    0.5
Name: proportion, dtype: float64


Baseline models: Logistic Regression and Random Forest


In [12]:
lr_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
])

rf_model = RandomForestClassifier(
    n_estimators=300,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

models = {
    "Logistic Regression": (lr_model, X_train, X_test),
    "Random Forest": (rf_model, X_train, X_test),
}

results = []

for name, (model, Xtr, Xte) in models.items():
    model.fit(Xtr, y_train)
    predictions = model.predict(Xte)

    accuracy = accuracy_score(y_test, predictions)
    balanced_acc = balanced_accuracy_score(y_test, predictions)
    f1 = f1_score(y_test, predictions, average="weighted", zero_division=0)

    results.append({
        "Model": name,
        "Accuracy": accuracy,
        "Balanced accuracy": balanced_acc,
        "Weighted F1": f1,
    })

    print(f"--- {name} ---")
    print(classification_report(y_test, predictions, zero_division=0))
    print("Confusion matrix (true labels are rows):")
    print(confusion_matrix(y_test, predictions))
    print(f"Accuracy:          {accuracy:.4f}")
    print(f"Balanced accuracy: {balanced_acc:.4f}")
    print(f"Weighted F1:       {f1:.4f}\n")

results_df = pd.DataFrame(results).set_index("Model")
display(results_df.round(4))

--- Logistic Regression ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1000
           1       1.00      1.00      1.00      1000

    accuracy                           1.00      2000
   macro avg       1.00      1.00      1.00      2000
weighted avg       1.00      1.00      1.00      2000

Confusion matrix (true labels are rows):
[[999   1]
 [  1 999]]
Accuracy:          0.9990
Balanced accuracy: 0.9990
Weighted F1:       0.9990

--- Random Forest ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1000
           1       1.00      1.00      1.00      1000

    accuracy                           1.00      2000
   macro avg       1.00      1.00      1.00      2000
weighted avg       1.00      1.00      1.00      2000

Confusion matrix (true labels are rows):
[[1000    0]
 [   0 1000]]
Accuracy:          1.0000
Balanced accuracy: 1.0000
Weighted F1:       1.0000



,Accuracy,Balanced accuracy,Weighted F1
Model,,,
Logistic Regression,0.999,0.999,0.999
Random Forest,1.000,1.000,1.000


Optional: cross-validation on the training partition


In [13]:
cv_folds = 5

for name, (model, _, _) in models.items():
    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=cv_folds,
        scoring="accuracy",
        n_jobs=-1,
    )
    print(
        f"{name} — {cv_folds}-fold training CV accuracy: "
        f"{scores.mean():.4f} ± {scores.std():.4f}"
    )

Logistic Regression — 5-fold training CV accuracy: 0.9951 ± 0.0016
Random Forest — 5-fold training CV accuracy: 1.0000 ± 0.0000
